In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [5]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",api_key=groq_api_key)

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage
model.invoke(
    [
        HumanMessage(content="Hi, my name is Ishaan and I am a chief Ai Engineer"),
    ]
)

AIMessage(content='Hello Ishaan! It’s great to meet a chief AI Engineer. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Ishaan and I am a chief Ai Engineer". Likely they want a response. There\'s no request. Should respond politely, maybe ask how can I help. No policy violation.'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 85, 'total_tokens': 161, 'completion_time': 0.157049363, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.004216152, 'prompt_tokens_details': None, 'queue_time': 0.349813363, 'total_time': 0.161265515}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_64b2f1c926', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fb20-8805-7320-b2f9-afa58892efb4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 76, 'total_tokens': 161, 'output_token_de

In [7]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, my name is Ishaan and I am a chief Ai Engineer"),
        AIMessage(content="Hi Ishaan, nice to meet you! How can I assist you today?"),
        HumanMessage(content="What is my name and what do I do?")
    ]
)   

AIMessage(content='Your name is **Ishaan**, and you work as a **Chief AI Engineer**.', additional_kwargs={'reasoning_content': 'User asks: "What is my name and what do I do?" We have the name Ishaan and role chief AI Engineer from earlier. So answer accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 121, 'total_tokens': 182, 'completion_time': 0.125472427, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.004566464, 'prompt_tokens_details': None, 'queue_time': 0.39696661, 'total_time': 0.130038891}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fb20-8cda-7141-aac9-4a44df0bb20d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 61, 'total_tokens': 182, 'output_token_details': {'reasoning': 33}})

###  Message History
We can use a Message history class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use it

In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]


with_message_history=RunnableWithMessageHistory(
    model,
    get_session_history
)


In [9]:
config={"configurable":{"session_id":"chat1"}}

In [10]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is Ishaan and I am a chief Ai Engineer"),
    ],
    config=config
)

/tmp/ipykernel_150422/1505388065.py:9: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id]=ChatMessageHistory()


In [11]:
with_message_history.invoke(
    [
        HumanMessage(content="What is my Name?"),
    ],
    config=config
)

AIMessage(content='Your name is **Ishaan**.', additional_kwargs={'reasoning_content': 'The user asks: "What is my Name?" We have context: user said "Hi, my name is Ishaan and I am a chief Ai Engineer". So answer: Ishaan. Should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 118, 'total_tokens': 179, 'completion_time': 0.126443707, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.006988038, 'prompt_tokens_details': None, 'queue_time': 0.333340977, 'total_time': 0.133431745}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_47082602e2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fb20-a5cd-79c0-8d99-875c1bde64e8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 61, 'total_tokens': 179, 'output_token_details': {'reasoning': 43}})

In [12]:
## change the session id to start a new conversation
config1={"configurable":{"session_id":"chat2"}}

response=with_message_history.invoke(
    [
        HumanMessage(content="What is my Name?"),
    ],
    config=config1
)
response.content

'I don’t have any information about your name. If you’d like to share it, feel free to let me know!'

In [15]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hey my name is John"),
    ],
    config=config1
)
response.content

'Nice to meet you, John! How can I help you today?'

In [16]:
response=with_message_history.invoke(
    [
        HumanMessage(content="What is my Name?"),
    ],
    config=config1
)
response.content

'Your name is John.'

### Prompt Templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case,the raw user input is just a message,Which we are passing to the LLM.Let's now make that a bit more complicated.First, let's add in a system message with some custom instructions(but still taking messages as input).Next,we'll add in more input besides just the messages.

In [15]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the questions to the nest of your ability."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain=prompt|model

In [16]:
chain.invoke(
    {"messages":[HumanMessage(content="Hi, my name is Ishaan")]}
)

AIMessage(content='Hello Ishaan! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Ishaan". Probably they want a greeting. We respond friendly. Ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 98, 'total_tokens': 151, 'completion_time': 0.111129505, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.014491622, 'prompt_tokens_details': None, 'queue_time': 0.320138353, 'total_time': 0.125621127}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fb25-463e-7452-8d42-44fbfc483fdd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 53, 'total_tokens': 151, 'output_token_details': {'reasoning': 28}})

In [17]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history
    )

In [18]:
response=config2={"configurable":{"session_id":"chat3"}}

response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is Ishaan"),
    ],
    config=config2
)
response.content   

'Hello Ishaan! Nice to meet you. How can I assist you today?'

In [19]:
## Add more complexity

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the questions to the nest of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain=prompt|model

In [20]:
response=chain.invoke(
    {"messages":[HumanMessage(content="Hi, my name is Ishaan")],"language":"Hindi"}
)
response.content

'नमस्ते इशान! आपसे मिलकर खुशी हुई। मैं आपकी कैसे मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in  a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [29]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
    )

/home/ishaan-yadav/Langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [30]:
config={"configurable":{"session_id":"chat4"}}

response=with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi I am Ishaan")],"language":"Hindi"},
    config =config
)
response.content

'नमस्ते ईशान! आपका स्वागत है। मैं आपकी किस प्रकार मदद कर सकता हूँ? यदि कोई सवाल या कोई खास जानकारी चाहिए, तो बेझिझक बताइए।'

### Manage the conversation History
One important concept to understand when bulding chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM, Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_message' -> helper reduces how many messages we're sending to the model. The trimmer allows us to specify how many tokens we  want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages.

In [13]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
                max_tokens=70,
                strategy="last",
                token_counter=model,
                include_system=True,
                allow_partial=False,
                start_on="human"
                )

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)


/home/ishaan-yadav/Langchain/venv/lib/python3.10/site-packages/langchain_core/language_models/base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
/home/ishaan-yadav/Langchain/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs

In [21]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt|model
)

response=chain.invoke(
    {
    "messages":messages+[HumanMessage(content="what ice cream do i like?")],
    "language":"English",
    }
)
response.content

'You mentioned that you like vanilla ice cream. 🍦'

In [22]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
    )
config={"configurable":{"session_id":"chat5"}}

In [23]:
response=with_message_history.invoke(
    {
        "messages":messages+[HumanMessage(content="what is my name?")],
        "language":"English",
    },
    config=config
)
response.content

'Your name is Bob.'